# Finite-size scaling — $h_c(L)$ and extrapolation
Fit BFFM $\\rho_Z$ and Rényi $S_2$ vs $h_z$ per $L$ with a symmetric tanh and an asymmetric Richards curve; $h_c$ = inflection. Then extrapolate $h_c(L)=h_c+b\\,L^{-x}$.

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
NQS = Path("../results/nqs")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

In [ ]:
def sweep(L):                                   # {hz: data} for finite (non-diverged) runs
    out = {}
    for p in sorted(NQS.glob(f"G-equiv_1_L{L}_hx0.00_hz*.json")):
        hz = float(re.search(r"hz([\d.]+)\.json", p.name).group(1))
        E = np.array([complex(x).real for x in json.load(open(p))["energy"]])
        if np.isfinite(E[-1]):
            out[hz] = json.load(open(p))
    return out

Ls   = [6, 8, 10, 12]
data = {L: sweep(L) for L in Ls}
cols = dict(zip(Ls, plt.cm.plasma(np.linspace(0, 0.85, len(Ls)))))

bffm_z = lambda d: d["order_params"]["WilsonBFFM"][-1][6] if d["order_params"]["WilsonBFFM"] else np.nan
renyi2 = lambda d: d["order_params"]["renyi2_entropy"][-1][0] if d["order_params"]["renyi2_entropy"] else np.nan
OBS = {"BFFM $\\rho_Z$": bffm_z, "Rényi $S_2$": renyi2}

def curve(L, fn):
    hz = np.array(sorted(data[L]))
    return hz, np.array([fn(data[L][h]) for h in hz])

# fit forms; h_c is the inflection point of each
tanh_f = lambda h, A, B, hc, w: A + B * np.tanh((h - hc) / w)
rich_f = lambda h, A, K, h0, w, nu: A + (K - A) / (1 + np.exp(-(h - h0) / w))**nu

def fit(x, y, form):
    """return (h_c, h_c_err, popt) — errors from the fit covariance (underestimated)."""
    if form == "tanh":
        p0 = [y.mean(), (y[-1] - y[0]) / 2, x[len(x)//2], 0.05]
        b  = ([-np.inf, -np.inf, x.min(), 1e-4], [np.inf, np.inf, x.max(), 1.0])
        popt, pcov = curve_fit(tanh_f, x, y, p0=p0, bounds=b, maxfev=40000)
        return popt[2], np.sqrt(pcov[2, 2]), popt
    p0 = [y[0], y[-1], x[len(x)//2], 0.03, 1.0]
    b  = ([-np.inf, -np.inf, x.min(), 1e-4, 1e-2], [np.inf, np.inf, x.max(), 1.0, 50])
    popt, pcov = curve_fit(rich_f, x, y, p0=p0, bounds=b, maxfev=60000)
    A, K, h0, w, nu = popt
    hc = h0 + w * np.log(nu)                      # inflection (NOT h0 when nu != 1)
    g  = np.array([0, 0, 1, np.log(nu), w / nu])  # d h_c / d(A,K,h0,w,nu)
    return hc, float(np.sqrt(g @ pcov @ g)), popt

In [ ]:
# --- knob: hz window used by the fits ---
HZ_FIT = (0.25, 0.50)      # (lo, hi); Richards needs ~>=8 points to be stable
FORMS  = {"tanh": tanh_f, "rich": rich_f}

In [ ]:
HC = {}                                          # (obs, form) -> {L: (h_c, err)}
fig, axes = plt.subplots(len(OBS), len(Ls), figsize=(3.2*len(Ls), 3.0*len(OBS)), squeeze=False)
for r, (oname, fn) in enumerate(OBS.items()):
    for c, L in enumerate(Ls):
        ax = axes[r][c]; hz, y = curve(L, fn)
        m = (hz >= HZ_FIT[0]) & (hz <= HZ_FIT[1]); x, yy = hz[m], y[m]
        ax.plot(hz, y, "o", color="0.7", ms=3)
        ax.plot(x, yy, "o", color=cols[L], ms=4)
        xs = np.linspace(x.min(), x.max(), 200)
        for form, ff, ls in [("tanh", tanh_f, "--"), ("rich", rich_f, "-")]:
            try:
                hc, err, popt = fit(x, yy, form)
                ax.plot(xs, ff(xs, *popt), ls, color=cols[L], lw=1.2)
                ax.axvline(hc, color=cols[L], ls=ls, lw=0.8, alpha=0.6)
                HC.setdefault((oname, form), {})[L] = (hc, err)
            except Exception as e:
                print(f"{oname} L={L} {form}: fit failed ({e})")
        if r == 0: ax.set_title(f"L={L}")
        if c == 0: ax.set_ylabel(oname)
        ax.set_xlabel("$h_z$")
fig.tight_layout()
# fig.savefig(FIG / "fss_fits.png", dpi=300, bbox_inches="tight")

In [ ]:
print(f"h_c(L) over hz in {HZ_FIT}  (— errors from fit covariance, underestimated)\n")
for (oname, form), d in HC.items():
    row = "  ".join(f"L{L}={d[L][0]:.4f}±{d[L][1]:.4f}" for L in Ls if L in d)
    print(f"{oname:14s} [{form:4s}]  {row}")

In [ ]:
# --- knob: fix the exponent x, or fit it (None) ---
FIX_X = None               # e.g. 1/0.629 ≈ 1.59 (3D-Ising nu); None -> fit x too

def fss(Ls_, hc, err, fix_x):
    iL = 1.0 / np.array(Ls_, float)
    s  = np.array(err) if np.all(np.isfinite(err)) and np.all(np.array(err) > 0) else None
    if fix_x is None:
        f = lambda t, hcinf, b, x: hcinf + b * t**x
        p0 = [min(hc), max(hc) - min(hc), 1.5]
    else:
        f = lambda t, hcinf, b: hcinf + b * t**fix_x
        p0 = [min(hc), max(hc) - min(hc)]
    popt, pcov = curve_fit(f, iL, hc, p0=p0, sigma=s, absolute_sigma=s is not None, maxfev=40000)
    return popt, np.sqrt(np.diag(pcov)), f

fig, ax = plt.subplots(figsize=(6, 4.5)); tt = np.linspace(0, 1/min(Ls), 200)
print(f"h_c(inf) from  h_c(L) = h_c + b·L^(-x)   [x {'fixed='+str(FIX_X) if FIX_X else 'fitted'}]\n")
for (oname, form), d in HC.items():
    Lk = [L for L in Ls if L in d]; hc = [d[L][0] for L in Lk]; er = [d[L][1] for L in Lk]
    popt, perr, f = fss(Lk, hc, er, FIX_X)
    lab = f"{oname} [{form}]"
    ax.errorbar(1/np.array(Lk), hc, yerr=er, fmt="o", ms=5, capsize=2, label=lab)
    ax.plot(tt, f(tt, *popt), "-", lw=1)
    x_txt = f", x={popt[2]:.2f}" if FIX_X is None else ""
    print(f"{lab:22s}  h_c(inf) = {popt[0]:.4f} ± {perr[0]:.4f}{x_txt}")
ax.set(xlabel="$1/L$", ylabel="$h_c(L)$", title="finite-size extrapolation")
ax.legend(fontsize=8)
fig.tight_layout()
# fig.savefig(FIG / "fss_extrapolation.png", dpi=300, bbox_inches="tight")